# Hands-on 1 · Applying LLMs to textual metadata



**Goal:** start with a title and abstract, retrieve candidate subject terms from a small local vocabulary, ask an LLM to select terms and a broad LCC class, then inspect the result. This demonstrates classification, prompt design, few-shot prompting, retrieval-augmented generation (RAG), and precision/recall. The example records and vocabulary below are **teaching examples**, not verified LCSH authority records or production cataloging.

**Run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all. No API key is needed. The first run downloads an open model (roughly a few GB), so allow several minutes. CPU is possible but slow.


Model: [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). Its outputs are suggestions for review, not catalog records.

In [ ]:
%pip -q install "transformers>=4.52,<5" "accelerate>=1.0,<2" "scikit-learn>=1.5,<2" "pandas>=2,<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import json, re, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE)
if DEVICE == "cpu": print("Tip: switch Colab to a T4 GPU for a faster live demo.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
model.eval()

def ask_llm(prompt, max_new_tokens=160):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0, inputs.shape[-1]:], skip_special_tokens=True).strip()

Device: cpu
Tip: switch Colab to a T4 GPU for a faster live demo.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 1. Prepare a book record

These records are synthetic, so we can share them safely. Edit `record` to try the second example after the first run. The `reference_terms` are an instructor-created comparison set, not authenticated LCSH or an independent gold standard.

In [ ]:
records = [
    {
        "title": "Farming in a Changing Climate",
        "abstract": "Examines how droughts, changing rainfall, and rising temperatures affect small farms. Discusses crop adaptation and sustainable agricultural practices.",
        "reference_terms": ["Climate change—Agricultural aspects", "Sustainable agriculture", "Droughts"],
    },
    {
        "title": "Museums and Civic Memory",
        "abstract": "Studies how museum exhibitions shape public memory and civic identity, using examples from national museums.",
        "reference_terms": ["Museums", "Collective memory", "Museum exhibits"],
    },
]
record = records[0]
print("Title:", record["title"])
print("Abstract:", record["abstract"])

Title: Farming in a Changing Climate
Abstract: Examines how droughts, changing rainfall, and rising temperatures affect small farms. Discusses crop adaptation and sustainable agricultural practices.


## 2. Try a basic prompt

The first response is unconstrained. Ask participants what would make it difficult to put directly into a catalog.

In [ ]:
basic_prompt = f"Assign subject headings and an LCC class to this book. Title: {record['title']}. Abstract: {record['abstract']}"
basic_response = ask_llm(basic_prompt)
print(basic_response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Subject Headings:
1. Agriculture - Drought-resistant crops
2. Agricultural sustainability
3. Climate change impacts on agriculture
4. Small-scale farming systems
5. Crop adaptation strategies

LCC Class (Library of Congress Classification):
G807 .F36 2023


## 3. Retrieve candidate terms (RAG)

This tiny local vocabulary stands in for an approved authority file. TF–IDF retrieves terms that share words with the record; the LLM then chooses only from those candidates. This is a teaching example of retrieval-augmented generation. For real cataloging, replace this list with your institution's verified authority data and review mappings with a cataloger.

In [ ]:
candidate_terms = [
    "Climate change—Agricultural aspects", "Agriculture—Environmental aspects",
    "Sustainable agriculture", "Droughts", "Farms", "Food security",
    "Museums", "National museums", "Collective memory", "Museum exhibits",
    "Civic engagement", "Rhetoric", "Public spaces", "Education",
]
descriptions = [term.replace("—", " ") for term in candidate_terms]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
term_matrix = vectorizer.fit_transform(descriptions)
query = record["title"] + " " + record["abstract"]
scores = cosine_similarity(vectorizer.transform([query]), term_matrix).ravel()
top_idx = [i for i in scores.argsort()[::-1] if scores[i] > 0][:8]
retrieved = [candidate_terms[i] for i in top_idx]
display(pd.DataFrame({"candidate": retrieved, "similarity": [round(float(scores[i]), 3) for i in top_idx]}))

,candidate,similarity
0,Droughts,0.447
1,Farms,0.447
2,Climate change—Agricultural aspects,0.344
3,Sustainable agriculture,0.270


## 4. Use a structured prompt

The broad LCC letters are included as allowed choices. The model does **not** produce a shelf-ready call number. Exact subject terms remain suggestions until a cataloger validates them against the authority file.

In [ ]:
lcc_choices = {"H": "Social sciences", "N": "Fine arts", "Q": "Science", "S": "Agriculture"}
candidate_lines = "\n".join(f"- {term}" for term in retrieved)
class_lines = "\n".join(f"- {letter}: {label}" for letter, label in lcc_choices.items())
structured_prompt = f"""You are assisting a librarian. Use only the supplied metadata.
Select up to three subject terms by copying them EXACTLY from the candidate list.
Choose one broad LCC letter from the allowed list. If evidence is inadequate, return an empty subject_terms list.
Return one JSON object only, with keys subject_terms (array of strings) and lcc_class (string).

Title: {record['title']}
Abstract: {record['abstract']}

Candidate subject terms:
{candidate_lines}

Allowed LCC classes:
{class_lines}"""
structured_response = ask_llm(structured_prompt, max_new_tokens=220)
print(structured_response)

```json
{
  "subject_terms": ["Droughts", "Farms", "Climate change—Agricultural aspects", "Sustainable agriculture"],
  "lcc_class": "H"
}
```


In [ ]:
def parse_and_check(raw, allowed_terms):
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return None, ["No JSON object found; inspect the raw response."]
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return None, [f"Invalid JSON: {exc}"]
    issues = []
    terms = data.get("subject_terms")
    if not isinstance(terms, list) or any(not isinstance(t, str) for t in terms):
        issues.append("subject_terms must be a list of strings")
        terms = []
    if len(terms) > 3: issues.append("More than three terms selected")
    unknown = [t for t in terms if t not in allowed_terms]
    if unknown: issues.append(f"Terms outside retrieved list: {unknown}")
    if data.get("lcc_class") not in lcc_choices:
        issues.append("LCC class is outside the allowed choices")
    return data, issues

prediction, issues = parse_and_check(structured_response, retrieved)
print("Parsed:", prediction)
print("Checks:", issues if issues else "Passed candidate and format checks")

Parsed: {'subject_terms': ['Droughts', 'Farms', 'Climate change—Agricultural aspects', 'Sustainable agriculture'], 'lcc_class': 'H'}
Checks: ['More than three terms selected']


## 5. Compare with an instructor reference set

Precision and recall below are **only for this one synthetic example**. Exact string matching penalizes legitimate variants and cannot assess subject appropriateness. The output needs human review even if the numbers look good.

In [ ]:
if prediction is None or issues:
    print("Fix or review the response before scoring. Raw output is shown above.")
else:
    predicted = set(prediction["subject_terms"])
    reference = set(record["reference_terms"])
    tp = len(predicted & reference)
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(reference) if reference else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}")
    print("Predicted:", sorted(predicted))
    print("Reference:", sorted(reference))

## 6. Participant exercise and wrap-up

1. Change `record = records[0]` to `records[1]`, then rerun cells 2–5. Does retrieval find the instructor's reference terms?
2. Edit the structured prompt: add one short example of a book and its selected terms (few-shot prompting), then rerun. Did the format or substance improve?
3. Discuss which errors come from retrieval, which from the LLM, and which require cataloger judgment. Why would a production workflow verify LCSH identifiers and record the source of AI suggestions?

